<h1>Bibliotecas</h1>

In [2]:
import requests
import pandas as pd
import os

from dotenv import load_dotenv
from datetime import datetime
from pathlib import Path

<h1>Google Fact Check</h1>

<h2>Consulta - API</h2>

In [6]:
load_dotenv('../google-factcheck-api-key.env')

API_KEY = os.getenv('GOOGLE_FACTCHECK_API_KEY')

if not API_KEY:
    raise ValueError('Chave não encontrada. Verifique em google-factcheck-api-key.env')

URL = 'https://factchecktools.googleapis.com/v1alpha1/claims:search'

# lista de termos
termos_busca = [
        'urnas eletrônicas',
        'TSE',
        'STF',
        'lula',
        'bolsonaro',
        'renan santos',
        'anistia',
        'eleições 2026',
        'presidente do brasil',
        'INSS',
        'escala 6x1',
        'MBL',
        'PT',
        'PL',
        'pix imposto'
]       

registros = []

# parametros
for termo in termos_busca:
    params = {
        'query': termo,
        'languageCode': 'pt',
        'pageSize': 10,
        'maxAgeDays': 720,
        'key': API_KEY
    }

    resposta = requests.get(URL, params=params)

    if resposta.status_code != 200:
        print(f'Erro ao buscar {termo}: {resposta.status_code}')
        print(resposta.text)
        continue

    dados = resposta.json()

    for claim in dados.get('claims', []):
        texto_claim = claim.get('text', '')
        data_claim = claim.get('claimDate', '')

        for review in claim.get('claimReview', []):
            registros.append({
                'termo_busca': termo,
                'texto_afirmacao': texto_claim,
                'data_claim': data_claim,
                'fonte': review.get('publisher', {}).get('name', ''),
                'url_checagem': review.get('url', ''),
                'avaliacao_original': review.get('textualRating', ''),
                'data_publicacao': review.get('reviewDate', '')
            })

df = pd.DataFrame(registros)

print(f'Total de Registros coletados: {len(df)}')

df.info()

Total de Registros coletados: 129
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 129 entries, 0 to 128
Data columns (total 7 columns):
 #   Column              Non-Null Count  Dtype 
---  ------              --------------  ----- 
 0   termo_busca         129 non-null    object
 1   texto_afirmacao     129 non-null    object
 2   data_claim          129 non-null    object
 3   fonte               129 non-null    object
 4   url_checagem        129 non-null    object
 5   avaliacao_original  129 non-null    object
 6   data_publicacao     129 non-null    object
dtypes: object(7)
memory usage: 7.2+ KB


<h2>Extração</h2>

In [11]:
data_agora = datetime.now().strftime('%Y-%m-%d_%H-%M-%S')

pasta_raw = Path('../dados/pipeline_falso_google_factcheck/raw')
pasta_raw.mkdir(parents=True, exist_ok=True)

caminho_saida = pasta_raw / f'dataset_google_factcheck_raw_{data_agora}.csv'

df.to_csv(caminho_saida, index=False, encoding="utf-8-sig")
print(f"Arquivo bruto salvo em: {caminho_saida}")

Arquivo bruto salvo em: ..\dados\pipeline_falso_google_factcheck\raw\dataset_google_factcheck_raw_2026-04-30_00-58-54.csv
